In [43]:
import yfinance as yf
import pandas as pd
import numpy as np
# import pandas_ta as ta
import os


class TradingStock():

    def __init__(self, ticker: str = None, period: str = '1y', interval: str = '1d', windows: list = None):
        """
        Parameters
        ----------
        ticker: stock ticker symbol
        windows: list of integers for MA window sizes (default: [20, 40, 80, 120])
        """
        self.ticker = ticker
        self.period = period
        self.windows = windows if windows else [20, 40, 80, 120]
        self.data = None



    def fetch(self):
        """
        Load Stock data by Ticker.
        """
        self.data = yf.Ticker(self.ticker).history(period=self.period)
        self.data.drop("Dividends", axis=1, inplace=True)
        self.data.dropna(subset=['Open', 'High', 'Low', 'Close', 'Volume'], inplace=True)
        return self


    def download_indicators(self):

        if self.data is not None and not self.data.empty:
            # Calculate and add indicators
            """
            self.data['ATR'] = self.data.ta.atr(length=20)
            self.data['RSI'] = self.data.ta.rsi()
            self.data['MidPrice'] = self.data.ta.midprice(length=1)
            macd = ta.macd(self.data['Close'])
            self.data = self.data.join(macd)

            get_slope = lambda col: linregress(np.arange(len(col)), np.array(col))[0]

            # Flexible MA and slope calculation
            for window in self.windows:
                ma_col = f"MA{window}"
                slope_col = f"slopeMA{window}"
                self.data[ma_col] = self.data.ta.sma(length=window)
                self.data[slope_col] = (
                    self.data[ma_col]
                    .rolling(window=window)
                    .apply(get_slope, raw=True)
                )
            """

            # Ensure directory exists
            os.makedirs("dataset", exist_ok=True)

            # Save to CSV file
            file_path = f"dataset/{self.ticker}_historical_data.csv"
            print(f"Saving data to {file_path}")
            self.data.to_csv(file_path, header=True, index=True)

aapl = TradingStock(ticker='AAPL', period='1y')
aapl.fetch().download_indicators()


Saving data to dataset/AAPL_historical_data.csv


In [46]:
def get_historical_data(ticker):
    """
    Parameters
    ----------
    ticker: stock ticker symbol
    """
    df = pd.read_csv(f'./dataset/{ticker}_historical_data.csv', parse_dates=['Date'])

    cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    df = df[cols]
    
    # Convert Date to string if required (to match image)
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.strftime('%Y-%m-%d')
    
    # Set Date as index
    df.set_index('Date', inplace=True)
    df = df.set_index(pd.DatetimeIndex(pd.to_datetime(df.index)))

    # Calculate 14-day Rate of Return: (Close_t - Close_{t-14}) / Close_{t-14}
    df['Rate of return_14days'] = (df['Close'] - df['Close'].shift(14)) / df['Close'].shift(14)
    df = df.dropna()
    df = df.round({
        'Open': 2, 'High': 2, 'Low': 2, 'Close': 2
    })

    return df

get_historical_data('AAPL').head(20)

,Open,High,Low,Close,Volume,Rate of return_14days
Date,,,,,,
2024-12-19,246.39,250.87,245.98,248.67,60882300,0.052501
2024-12-20,246.92,253.85,244.58,253.34,147495300,0.062190
2024-12-23,253.62,254.50,252.31,254.12,40858800,0.052009
2024-12-24,254.34,257.05,254.14,257.04,23234700,0.062508
2024-12-26,257.03,258.93,256.47,257.85,27237100,0.065750
2024-12-27,256.67,257.54,251.92,254.44,42355300,0.052504
2024-12-30,251.09,252.36,249.62,251.06,35557500,0.022087
2024-12-31,251.30,252.14,248.31,249.29,39480700,0.010695
2025-01-02,247.81,247.98,240.73,242.75,55740700,-0.010710


In [ ]:
def generate_mean_reversion_signals(ticker, rsi_threshold=30, window=20):
    df = get_historical_data(ticker)

    # Calculate 20-day moving average as mean
    df['MA20'] = df['Close'].rolling(window=window).mean()

    # Calculate Bollinger Bands
    rolling_std = df['Close'].rolling(window=window).std()
    df['Lower_BB'] = df['MA20'] - (2 * rolling_std)
    df['Upper_BB'] = df['MA20'] + (2 * rolling_std)

    # Calculate RSI
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs = avg_gain / avg_loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # Generate signals for backtesting (1 = buy, 0 = hold, -1 = sell)
    df['signal'] = 0
    # Buy condition: Close below Lower Bollinger Band AND RSI below threshold
    df.loc[(df['Close'] < df['Lower_BB']) & (df['RSI'] < rsi_threshold), 'signal'] = 1

    # Sell condition: Close above Upper Bollinger Band AND RSI above (100 - threshold)
    df.loc[(df['Close'] > df['Upper_BB']) & (df['RSI'] > (100 - rsi_threshold)), 'signal'] = -1

    df.dropna(inplace=True)
    # For backtest, you often want to convert signals to order instructions
    # e.g., 1 for buy, 0 for no action
    return df

generate_mean_reversion_signals('AAPL').value_counts('signal')

signal
 0    219
 1      8
-1      4
Name: count, dtype: int64

In [42]:
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

from backtesting.test import SMA, GOOG


class SmaCross(Strategy):
    def init(self):
        price = self.data.Close
        self.ma1 = self.I(SMA, price, 30)
        self.ma2 = self.I(SMA, price, 60)

    def next(self):
        if crossover(self.ma1, self.ma2):
            self.buy()
        elif crossover(self.ma2, self.ma1):
            self.sell()

df = get_historical_data('AAPL')
bt = Backtest(df, SmaCross, commission=.002,
              exclusive_orders=True)
stats = bt.run()
bt.plot()

/var/folders/6x/jgzf8_717bd_yypnq8xtrhw40000gn/T/ipykernel_17491/1467149822.py:22: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


GridPlot(id='p2116', ...)

In [15]:
df = yf.download('MSFT', period='1y', interval='1d')
df = df.droplevel(level=1, axis=1)
df.head()

/var/folders/6x/jgzf8_717bd_yypnq8xtrhw40000gn/T/ipykernel_3197/1799860941.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('MSFT', period='1y', interval='1d')
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2024-10-21,415.659424,415.838075,410.666907,413.019241,14206100
2024-10-22,424.324402,427.371503,414.924967,415.371595,25482200
2024-10-23,421.436066,427.867760,419.381483,427.649398,19654400
2024-10-24,421.565094,422.805779,419.252439,422.160599,13581600
2024-10-25,424.959595,429.297026,423.391382,423.579968,16899100


In [ ]:
df['MA05'] = df.ta.sma(length=5)
df['MA20'] = df.ta.sma(length=20)
df['MA60'] = df.ta.sma(length=60)
df['RSI'] = df.ta.rsi(length=14)
macd = df.ta.macd(fast=12, slow=26, signal=9)
df = df.join(macd)
df.tail()

,Close,High,Low,Open,Volume,MA05,MA20,MA60,RSI,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9
Date,,,,,,,,,,,,
2025-10-15,513.429993,517.190002,510.000000,514.960022,14694700,514.882001,515.792998,513.316227,49.009710,1.679366,-0.813811,2.493177
2025-10-16,511.609985,516.849976,508.130005,512.580017,15559600,512.723993,515.950996,513.425730,47.221272,1.189377,-1.043040,2.232417
2025-10-17,513.580017,515.479980,507.309998,509.040009,19867800,513.247998,515.733498,513.484705,49.374765,0.949082,-1.026668,1.975750
2025-10-20,516.789978,518.700012,513.429993,514.609985,14656200,513.795996,515.850496,513.550089,52.757214,1.006066,-0.775747,1.781813
2025-10-21,514.099915,518.690002,513.039978,517.760010,7425808,513.901978,516.093991,513.590773,49.756911,0.824655,-0.765727,1.590382


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/brianlee/Documents/FYP-project/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/brianlee/Documents/FYP-project/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 339, in dispatch_control
    await self.process_control(msg)
  File "/Users/brianlee/Documents/FYP-project/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 345, in process_control
    idents, msg = self.session.feed_identities(msg, copy=False)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/brianlee/Documents/FYP-project/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/brian